In [1]:
from ctrader_open_api import Client, Protobuf, TcpProtocol, Auth, EndPoints
from ctrader_open_api.messages.OpenApiCommonMessages_pb2 import *
from ctrader_open_api.messages.OpenApiMessages_pb2 import *
from ctrader_open_api.messages.OpenApiModelMessages_pb2 import *
from twisted.internet import reactor
import json
import datetime
import calendar
import keyring

:0: UserWarning: You do not have a working installation of the service_identity module: 'No module named 'service_identity''.  Please install it from <https://pypi.python.org/pypi/service_identity> and make sure all of its dependencies are satisfied.  Without the service_identity module, Twisted can perform only rudimentary TLS client hostname verification.  Many valid certificate/hostname mappings may be rejected.


In [2]:
credentialsFile = open("credentials-dev.json")
credentials = json.load(credentialsFile)
credentials['Secret'] = keyring.get_password("ctrader", credentials['ClientId'])

In [3]:
host = EndPoints.PROTOBUF_LIVE_HOST if credentials["HostType"].lower() == "live" else EndPoints.PROTOBUF_DEMO_HOST
client = Client(host, EndPoints.PROTOBUF_PORT, TcpProtocol)

In [4]:
symbolName = "US500"

In [5]:
dailyBars = []

In [6]:
def transformTrendbar(trendbar):
    openTime = datetime.datetime.fromtimestamp(trendbar.utcTimestampInMinutes * 60, datetime.timezone.utc)
    openPrice = (trendbar.low + trendbar.deltaOpen) / 100000.0
    highPrice = (trendbar.low + trendbar.deltaHigh) / 100000.0
    lowPrice = trendbar.low / 100000.0
    closePrice = (trendbar.low + trendbar.deltaClose) / 100000.0
    return [openTime, openPrice, highPrice, lowPrice, closePrice, trendbar.volume]

In [7]:
def trendbarsResponseCallback(result):
    print("\nTrendbars received")
    trendbars = Protobuf.extract(result)
    barsData = list(map(transformTrendbar, trendbars.trendbar))
    global dailyBars
    dailyBars.clear()
    dailyBars.extend(barsData)
    print("\ndailyBars length:", len(dailyBars))
    print("\Stopping reactor...")
    reactor.stop()

def symbolsResponseCallback(result):
    print("\nSymbols received")
    symbols = Protobuf.extract(result)
    global symbolName
    symbolsFilterResult = list(filter(lambda symbol: symbol.symbolName == symbolName, symbols.symbol))
    if len(symbolsFilterResult) == 0:
        raise Exception(f"There is symbol that matches to your defined symbol name: {symbolName}")
    elif len(symbolsFilterResult) > 1:
        raise Exception(f"More than one symbol matched with your defined symbol name: {symbolName}, match result: {symbolsFilterResult}")
    symbol = symbolsFilterResult[0]

    # Fetch multiple chunks
    num_chunks = 52*10
    weeks_per_chunk = 1
    now = datetime.datetime.utcnow()
    requests = []
    for i in range(num_chunks):
        to_time = now - datetime.timedelta(weeks=weeks_per_chunk * i)
        from_time = to_time - datetime.timedelta(weeks=weeks_per_chunk)
        request = ProtoOAGetTrendbarsReq()
        request.symbolId = symbol.symbolId
        request.ctidTraderAccountId = credentials["AccountId"]
        request.period = ProtoOATrendbarPeriod.M1
        request.fromTimestamp = int(calendar.timegm(from_time.utctimetuple())) * 1000
        request.toTimestamp = int(calendar.timegm(to_time.utctimetuple())) * 1000
        requests.append(request)

    # Helper to chain requests
    def fetch_next(index):
        if index >= len(requests):
            print("\nAll chunks fetched")
            reactor.stop()
            return
        deferred = client.send(requests[index])
        def on_success(result):
            trendbars = Protobuf.extract(result)
            barsData = list(map(transformTrendbar, trendbars.trendbar))
            global dailyBars
            dailyBars.extend(barsData)
            print(f"\nFetched chunk {index+1}/{len(requests)}, bars: {len(barsData)}")
            fetch_next(index + 1)
        deferred.addCallbacks(on_success, onError)

    # Start fetching
    global dailyBars
    dailyBars.clear()
    fetch_next(0)
    
def accountAuthResponseCallback(result):
    print("\nAccount authenticated")
    request = ProtoOASymbolsListReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.includeArchivedSymbols = False
    deferred = client.send(request)
    deferred.addCallbacks(symbolsResponseCallback, onError)
    
def applicationAuthResponseCallback(result):
    print("\nApplication authenticated")
    request = ProtoOAAccountAuthReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.accessToken = credentials["AccessToken"]
    deferred = client.send(request)
    deferred.addCallbacks(accountAuthResponseCallback, onError)

def onError(client, failure): # Call back for errors
    print("\nMessage Error: ", failure)

def disconnected(client, reason): # Callback for client disconnection
    print("\nDisconnected: ", reason)

def onMessageReceived(client, message): # Callback for receiving all messages
    if message.payloadType in [ProtoHeartbeatEvent().payloadType, ProtoOAAccountAuthRes().payloadType, ProtoOAApplicationAuthRes().payloadType, ProtoOASymbolsListRes().payloadType, ProtoOAGetTrendbarsRes().payloadType]:
        return
    print("\nMessage received: \n", Protobuf.extract(message))
    
def connected(client): # Callback for client connection
    print("\nConnected")
    request = ProtoOAApplicationAuthReq()
    request.clientId = credentials["ClientId"]
    request.clientSecret = credentials["Secret"]
    deferred = client.send(request)
    deferred.addCallbacks(applicationAuthResponseCallback, onError)
    
# Setting optional client callbacks
client.setConnectedCallback(connected)
client.setDisconnectedCallback(disconnected)
client.setMessageReceivedCallback(onMessageReceived)

In [8]:
# Starting the client service
client.startService()

# Run Twisted reactor, we imported it earlier
reactor.run()


Connected

Application authenticated

Account authenticated

Symbols received

Fetched chunk 1/520, bars: 6884

Fetched chunk 2/520, bars: 6649

Fetched chunk 3/520, bars: 6888

Fetched chunk 4/520, bars: 6887

Fetched chunk 5/520, bars: 6880

Fetched chunk 6/520, bars: 6629

Fetched chunk 7/520, bars: 6869

Fetched chunk 8/520, bars: 5319

Fetched chunk 9/520, bars: 5204

Fetched chunk 10/520, bars: 6869

Fetched chunk 11/520, bars: 6891

Fetched chunk 12/520, bars: 6845

Fetched chunk 13/520, bars: 5771

Fetched chunk 14/520, bars: 6889

Fetched chunk 15/520, bars: 6889

Fetched chunk 16/520, bars: 6892

Fetched chunk 17/520, bars: 6825

Fetched chunk 18/520, bars: 6867

Fetched chunk 19/520, bars: 6885

Fetched chunk 20/520, bars: 6838

Fetched chunk 21/520, bars: 6776

Fetched chunk 22/520, bars: 6804

Fetched chunk 23/520, bars: 6810

Fetched chunk 24/520, bars: 6761

Fetched chunk 25/520, bars: 6801

Fetched chunk 26/520, bars: 6565

Fetched chunk 27/520, bars: 6818

Fetched chu

In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.DataFrame(np.array(dailyBars),
                   columns=['Time', 'Open', 'High', 'Low', 'Close', 'Volume']).drop_duplicates().reset_index(drop=True)
df["Open"] = pd.to_numeric(df["Open"])
df["High"] = pd.to_numeric(df["High"])
df["Low"] = pd.to_numeric(df["Low"])
df["Close"] = pd.to_numeric(df["Close"])
df["Volume"] = pd.to_numeric(df["Volume"])

In [11]:
df['Time'].describe()

count                             3205696
mean     2021-06-20 08:32:59.543162+00:00
min             2016-03-09 04:04:00+00:00
25%             2019-03-15 09:29:45+00:00
50%             2021-07-26 09:42:30+00:00
75%             2023-10-27 15:57:15+00:00
max             2026-02-25 04:03:00+00:00
Name: Time, dtype: object

In [12]:
df = df.sort_values('Time')
df = df.drop_duplicates().reset_index(drop=True)
df.to_csv(f'../../data/{symbolName}_1minute.csv', index=False)

In [13]:
df

,Time,Open,High,Low,Close,Volume
0,2016-03-09 04:04:00+00:00,1983.3,1983.3,1983.3,1983.3,2
1,2016-03-09 04:06:00+00:00,1983.4,1983.4,1983.3,1983.3,8
2,2016-03-09 04:13:00+00:00,1983.1,1983.1,1983.1,1983.1,2
3,2016-03-09 04:19:00+00:00,1983.2,1983.2,1983.2,1983.2,2
4,2016-03-09 04:20:00+00:00,1983.0,1983.0,1982.8,1982.8,4
...,...,...,...,...,...,...
3205691,2026-02-25 03:59:00+00:00,6898.7,6899.3,6898.1,6898.3,54
3205692,2026-02-25 04:00:00+00:00,6898.1,6898.3,6896.8,6897.3,79
3205693,2026-02-25 04:01:00+00:00,6897.2,6897.2,6895.3,6895.8,91
3205694,2026-02-25 04:02:00+00:00,6895.7,6896.1,6894.3,6894.8,85
